# Single-Mask DF-DPC — Calculadora de parámetros de laboratorio

Implementa las condiciones geométricas descritas en el paper (Sección EXPERIMENT) y devuelve los rangos permitidos para:

- Distancia fuente–máscara $d_{sm}$
- Distancia máscara–detector $d_{md}$
- Distancia fuente–detector total $L = d_{sm}+d_{md}$ (con tope de laboratorio, por defecto **0.7 m**)
- Distancia muestra–detector $d_{od}$ (muestra apoyada contra la máscara)
- Magnificación geométrica $M$
- Pasos de *dithering* (sub-período de máscara)
- Resolución limitada por foco y *pixel size* efectivo a la muestra

### Condiciones del paper
El alineamiento exige que la proyección de la máscara sobre el detector tenga un período entero $N$ veces el pitch del detector:
$$ p_m \cdot M \;=\; N \cdot p_d $$
donde
- $N=2$ → configuración **DPC** o **DF** (M ≈ 2×)
- $N=3$ → configuración **DF-DPC** (M ≈ 3×)

y la magnificación es $M = (d_{sm}+d_{md})/d_{sm}$, lo que fija el cociente entre distancias una vez elegida $N$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Parámetros de laboratorio

Los defaults coinciden con los del paper. Editá libremente la celda siguiente con los valores de tu instalación.

In [ ]:
@dataclass
class LabSetup:
    # --- Máscara ---
    p_mask_um: float = 53.0       # período de la máscara [µm]
    slit_um: float = 20.0         # ancho de slit (apertura) [µm]
    au_thickness_um: float = 100. # espesor del oro [µm]

    # --- Detector ---
    p_det_um: float = 55.0        # pixel pitch [µm]
    det_w_mm: float = 70.0        # ancho activo [mm]
    det_h_mm: float = 14.0        # alto activo  [mm]

    # --- Fuente ---
    focal_spot_um: float = 20.0   # diámetro nominal del foco [µm] (Hamamatsu: 7/20/50)
    kV: float = 40.0              # tensión del tubo [kV]

    # --- Restricciones del banco ---
    L_max_m: float = 0.70         # distancia fuente-detector máxima [m]
    L_min_m: float = 0.20         # distancia fuente-detector mínima razonable [m]

    # --- Adquisición ---
    config: str = 'DF-DPC'        # 'DPC', 'DF' o 'DF-DPC'
    n_dither: int = 8             # pasos de dithering por período de máscara

lab = LabSetup()
lab

## 2. Geometría según la configuración

Dada $N$ ($=2$ para DPC/DF, $=3$ para DF-DPC) y los pitches, queda fijada la magnificación nominal
$$M^\star = \frac{N\,p_d}{p_m}$$
y, para cualquier $L \le L_{max}$, las distancias quedan determinadas:
$$d_{sm} = \frac{L}{M^\star}, \qquad d_{md} = L\,\frac{M^\star-1}{M^\star}.$$

In [ ]:
_N_BY_CONFIG = {'DPC': 2, 'DF': 2, 'DF-DPC': 3}

def nominal_magnification(lab: LabSetup, config: Optional[str] = None) -> float:
    cfg = (config or lab.config).upper()
    if cfg not in _N_BY_CONFIG:
        raise ValueError(f"config debe ser uno de {list(_N_BY_CONFIG)}")
    N = _N_BY_CONFIG[cfg]
    return N * lab.p_det_um / lab.p_mask_um

def distances_from_L(L_m: float, M: float):
    """Dada L = d_sm + d_md y M, devuelve (d_sm, d_md) en metros."""
    d_sm = L_m / M
    d_md = L_m - d_sm
    return d_sm, d_md

for cfg in ['DPC', 'DF-DPC']:
    M = nominal_magnification(lab, cfg)
    print(f"{cfg:7s}  N={_N_BY_CONFIG[cfg]}   M* = N·p_d/p_m = {M:.4f}")
    print(f"          p_mask·M* = {lab.p_mask_um*M:.2f} µm  vs  N·p_det = {_N_BY_CONFIG[cfg]*lab.p_det_um:.2f} µm")

## 3. Rangos permitidos

Barrido de $L = d_{sm} + d_{md}$ entre $L_{min}$ y $L_{max}$, manteniendo la condición de alineamiento $p_m M = N p_d$.

In [ ]:
def sweep_geometry(lab: LabSetup, config: Optional[str] = None, n: int = 200):
    M = nominal_magnification(lab, config)
    L = np.linspace(lab.L_min_m, lab.L_max_m, n)
    d_sm, d_md = distances_from_L(L, M)
    # Muestra apoyada contra la máscara → d_od ≈ d_md
    d_od = d_md
    # Magnificación de la muestra al detector
    M_obj = (d_sm + d_od) / d_sm  # ≈ M cuando muestra está pegada a la máscara
    # Pixel efectivo a la muestra (campo proyectado)
    px_eff_um = lab.p_det_um / M_obj
    # Penumbra por tamaño de foco a la muestra (proyectada al detector)
    blur_det_um = lab.focal_spot_um * d_od / d_sm
    blur_obj_um = blur_det_um / M_obj  # referida al plano de la muestra
    # Campo de visión (FOV) a la muestra
    fov_w_mm = lab.det_w_mm / M_obj
    fov_h_mm = lab.det_h_mm / M_obj
    return dict(L=L, d_sm=d_sm, d_md=d_md, d_od=d_od, M=np.full_like(L, M),
                M_obj=M_obj, px_eff_um=px_eff_um,
                blur_det_um=blur_det_um, blur_obj_um=blur_obj_um,
                fov_w_mm=fov_w_mm, fov_h_mm=fov_h_mm)

def summary_table(lab: LabSetup):
    rows = []
    for cfg in ['DPC', 'DF-DPC']:
        s = sweep_geometry(lab, cfg, n=2)
        for i, tag in enumerate(['min', 'max']):
            rows.append([
                cfg, tag,
                f"{s['L'][i]*100:.1f}",
                f"{s['d_sm'][i]*100:.1f}",
                f"{s['d_md'][i]*100:.1f}",
                f"{s['M'][i]:.3f}",
                f"{s['px_eff_um'][i]:.2f}",
                f"{s['blur_obj_um'][i]:.2f}",
                f"{s['fov_w_mm'][i]:.1f} × {s['fov_h_mm'][i]:.1f}",
            ])
    hdr = ['cfg','L', 'L[cm]', 'd_sm[cm]', 'd_md[cm]', 'M', 'px_eff[µm]', 'blur_obj[µm]', 'FOV[mm]']
    width = [max(len(h), max(len(r[i]) for r in rows)) for i, h in enumerate(hdr)]
    line = '  '.join(h.ljust(width[i]) for i, h in enumerate(hdr))
    print(line)
    print('-'*len(line))
    for r in rows:
        print('  '.join(str(c).ljust(width[i]) for i,c in enumerate(r)))

summary_table(lab)

## 4. Punto de operación: ¿qué $L$ elegir?

Para una distancia fuente-detector total $L$ (m), devuelve todos los parámetros derivados.

In [ ]:
def operating_point(lab: LabSetup, L_m: float, config: Optional[str] = None):
    cfg = (config or lab.config).upper()
    if L_m > lab.L_max_m + 1e-9:
        raise ValueError(f"L={L_m*100:.1f} cm excede el tope del banco ({lab.L_max_m*100:.1f} cm)")
    if L_m < lab.L_min_m:
        raise ValueError(f"L={L_m*100:.1f} cm por debajo del mínimo razonable ({lab.L_min_m*100:.1f} cm)")
    M = nominal_magnification(lab, cfg)
    d_sm, d_md = distances_from_L(L_m, M)
    d_od = d_md  # muestra contra la máscara
    M_obj = (d_sm + d_od) / d_sm
    px_eff_um = lab.p_det_um / M_obj
    blur_det_um = lab.focal_spot_um * d_od / d_sm
    blur_obj_um = blur_det_um / M_obj
    # Dithering: paso a la máscara y proyectado a la muestra
    step_mask_um = lab.p_mask_um / lab.n_dither
    step_obj_um  = step_mask_um  # la muestra está en el plano de la máscara
    step_det_um  = step_mask_um * M  # equivalencia en el detector
    # Muestreo efectivo tras dithering
    eff_sampling_um = px_eff_um / lab.n_dither * (lab.p_mask_um*M/lab.p_det_um/_N_BY_CONFIG[cfg])
    print(f"Configuración: {cfg}   (N = {_N_BY_CONFIG[cfg]})")
    print(f"  L (fuente-detector)  : {L_m*100:6.2f} cm   [tope: {lab.L_max_m*100:.1f} cm]")
    print(f"  d_sm (fuente-máscara): {d_sm*100:6.2f} cm")
    print(f"  d_md (máscara-det.)  : {d_md*100:6.2f} cm")
    print(f"  d_od (muestra-det.)  : {d_od*100:6.2f} cm   (muestra contra máscara)")
    print(f"  Magnificación M      : {M:.4f}")
    print(f"  Período proyectado   : {lab.p_mask_um*M:6.2f} µm  ({_N_BY_CONFIG[cfg]} × p_det = {_N_BY_CONFIG[cfg]*lab.p_det_um:.2f} µm)")
    print()
    print(f"  Pixel efectivo en muestra : {px_eff_um:6.2f} µm/px")
    print(f"  Penumbra (foco {lab.focal_spot_um:.0f} µm):")
    print(f"       en detector  : {blur_det_um:6.2f} µm")
    print(f"       en muestra   : {blur_obj_um:6.2f} µm")
    print(f"  FOV en muestra   : {lab.det_w_mm/M_obj:5.1f} × {lab.det_h_mm/M_obj:5.1f} mm")
    print()
    print(f"  Dithering ({lab.n_dither} pasos / período):")
    print(f"       paso en máscara/muestra : {step_obj_um:6.2f} µm")
    print(f"       paso equivalente en det.: {step_det_um:6.2f} µm")
    return dict(cfg=cfg, M=M, d_sm=d_sm, d_md=d_md, d_od=d_od,
                px_eff_um=px_eff_um, blur_obj_um=blur_obj_um,
                step_mask_um=step_mask_um, step_det_um=step_det_um)

_ = operating_point(lab, L_m=0.65, config='DF-DPC')

## 5. Visualización del barrido

In [ ]:
def plot_sweep(lab: LabSetup):
    fig, ax = plt.subplots(2, 2, figsize=(11, 7.5))
    for cfg, color in [('DPC', 'tab:blue'), ('DF-DPC', 'tab:orange')]:
        s = sweep_geometry(lab, cfg, n=200)
        L_cm = s['L']*100
        ax[0,0].plot(L_cm, s['d_sm']*100, color=color, label=f"{cfg}  d_sm")
        ax[0,0].plot(L_cm, s['d_md']*100, color=color, ls='--', label=f"{cfg}  d_md")
        ax[0,1].plot(L_cm, s['M'], color=color, label=f"{cfg}  M = {s['M'][0]:.3f}")
        ax[1,0].plot(L_cm, s['px_eff_um'], color=color, label=cfg)
        ax[1,1].plot(L_cm, s['blur_obj_um'], color=color, label=cfg)
    for a in ax.ravel():
        a.set_xlabel('L = d_sm + d_md  [cm]')
        a.grid(alpha=.3); a.legend(fontsize=8)
    ax[0,0].set_ylabel('distancia [cm]'); ax[0,0].set_title('Distancias fuente-máscara / máscara-det.')
    ax[0,1].set_ylabel('M'); ax[0,1].set_title('Magnificación nominal')
    ax[1,0].set_ylabel('µm/px en muestra'); ax[1,0].set_title('Pixel efectivo (p_det / M)')
    ax[1,1].set_ylabel('µm'); ax[1,1].set_title(f"Penumbra en muestra (foco {lab.focal_spot_um:.0f} µm)")
    fig.suptitle(f"Single-Mask DF-DPC  |  p_mask={lab.p_mask_um} µm, p_det={lab.p_det_um} µm, L_max={lab.L_max_m*100:.0f} cm")
    fig.tight_layout()
    return fig

plot_sweep(lab);

## 6. Diagrama de la geometría seleccionada

In [ ]:
def plot_geometry(lab: LabSetup, L_m: float, config: Optional[str] = None):
    cfg = (config or lab.config).upper()
    M = nominal_magnification(lab, cfg)
    d_sm, d_md = distances_from_L(L_m, M)
    fig, ax = plt.subplots(figsize=(11, 3.2))
    y0 = 0
    # eje
    ax.hlines(y0, 0, L_m*100, color='lightgray', lw=1)
    # fuente
    ax.plot(0, y0, 'o', color='gold', markersize=14, markeredgecolor='k')
    ax.text(0, y0+0.18, f"Fuente\n({lab.focal_spot_um:.0f} µm, {lab.kV:.0f} kV)", ha='center', fontsize=8)
    # máscara
    xm = d_sm*100
    ax.vlines(xm, y0-0.12, y0+0.12, color='tab:orange', lw=4)
    ax.text(xm, y0+0.18, f"Máscara\np={lab.p_mask_um:.0f} µm\nslit={lab.slit_um:.0f} µm", ha='center', fontsize=8)
    # muestra (justo después de la máscara)
    ax.vlines(xm+0.4, y0-0.08, y0+0.08, color='tab:green', lw=3)
    ax.text(xm+0.4, y0-0.28, 'Muestra', ha='center', fontsize=8, color='tab:green')
    # detector
    xd = (d_sm+d_md)*100
    ax.vlines(xd, y0-0.15, y0+0.15, color='tab:blue', lw=5)
    ax.text(xd, y0+0.21, f"Detector\np={lab.p_det_um:.0f} µm", ha='center', fontsize=8)
    # cotas
    ax.annotate('', xy=(xm, y0-0.35), xytext=(0, y0-0.35), arrowprops=dict(arrowstyle='<->'))
    ax.text(xm/2, y0-0.42, f"d_sm = {d_sm*100:.1f} cm", ha='center', fontsize=9)
    ax.annotate('', xy=(xd, y0-0.35), xytext=(xm, y0-0.35), arrowprops=dict(arrowstyle='<->'))
    ax.text((xm+xd)/2, y0-0.42, f"d_md = {d_md*100:.1f} cm", ha='center', fontsize=9)
    ax.annotate('', xy=(xd, y0+0.55), xytext=(0, y0+0.55), arrowprops=dict(arrowstyle='<->'))
    ax.text(xd/2, y0+0.6, f"L = {L_m*100:.1f} cm   (M = {M:.3f}, {cfg})", ha='center', fontsize=9, weight='bold')
    ax.set_ylim(-0.6, 0.85); ax.set_xlim(-3, L_m*100+3)
    ax.set_yticks([]); ax.set_xlabel('posición a lo largo del eje óptico [cm]')
    ax.set_title(f"Geometría {cfg}")
    fig.tight_layout()
    return fig

plot_geometry(lab, L_m=0.65, config='DF-DPC');

## 7. Validación: ¿la condición de Moiré nulo es alcanzable?

El método de alineamiento del paper exige que el período proyectado $p_m M$ sea exactamente $N p_d$. Como $p_m$ y $p_d$ son fijos, $M^\star$ queda fijo y por lo tanto **el cociente $d_{sm}/d_{md}$ está fijo**; sólo se puede mover el sistema en $L$. Si $M^\star$ está lejos de 2 o 3, la configuración no es físicamente alineable con esa $N$.

In [ ]:
def check_feasibility(lab: LabSetup, tol: float = 0.15):
    ok = []
    for cfg, N in _N_BY_CONFIG.items():
        if cfg == 'DF':  # DF y DPC comparten N=2; lo mostramos una sola vez
            continue
        M = nominal_magnification(lab, cfg)
        deviation = abs(M - N) / N
        status = 'OK' if deviation <= tol else 'REVISAR'
        ok.append((cfg, N, M, deviation, status))
        print(f"{cfg:7s}  M* = {M:.3f}   target N = {N}   desvío = {deviation*100:5.2f}%   → {status}")
    # comprobar tope de L
    print()
    for cfg, _, M, *_ in ok:
        # para que la muestra quede al menos a 'd_od_min' de la máscara, necesitamos L razonable
        Lmin_needed = lab.L_min_m
        d_sm_max, d_md_max = distances_from_L(lab.L_max_m, M)
        print(f"{cfg:7s}  con L = L_max = {lab.L_max_m*100:.1f} cm  →  d_sm = {d_sm_max*100:.1f} cm, d_md = {d_md_max*100:.1f} cm")

check_feasibility(lab)

## 8. Test rápido con tus valores

Editá la celda y volvé a ejecutar.

In [ ]:
mi_lab = LabSetup(
    p_mask_um   = 53.0,
    slit_um     = 20.0,
    p_det_um    = 55.0,
    det_w_mm    = 70.0,
    det_h_mm    = 14.0,
    focal_spot_um = 20.0,
    kV          = 40.0,
    L_max_m     = 0.70,
    L_min_m     = 0.25,
    config      = 'DF-DPC',
    n_dither    = 8,
)

summary_table(mi_lab)
print()
check_feasibility(mi_lab)
print()
operating_point(mi_lab, L_m=mi_lab.L_max_m, config='DF-DPC')
plot_sweep(mi_lab)
plot_geometry(mi_lab, L_m=mi_lab.L_max_m, config='DF-DPC');